# node21det: executor no Colab

Roda uma execução do protocolo. Ambiente: **GPU** (Ambiente de execução → Alterar o tipo → T4 ou L4).

Rode as células em ordem. Se a sessão cair, abra de novo e rode tudo: o treino
retoma do `last.pt` salvo no Drive.

## 1. Configuração

In [ ]:
# Pasta do Drive com images/ e metadata.csv
DRIVE_DATA = "/content/drive/MyDrive/dataset_node21/cxr_images/proccessed_data"
# Onde ficam checkpoints e históricos (persistem entre sessões)
DRIVE_RUNS = "/content/drive/MyDrive/tcc/runs"

CONFIG = "configs/frcnn_none.yaml"
RUN_NAME = "pilot_frcnn_none"
EXTRA_ARGS = "--mode select --max-epochs 2"   # piloto: 2 épocas. Execução completa: "--mode select"

DESCONECTAR_AO_FIM = False   # True para liberar a GPU (e parar de gastar unidades) quando terminar

## 2. GPU e Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## 3. Código

In [ ]:
import os
if os.path.exists("/content/node21det"):
    !git -C /content/node21det pull -q
else:
    !git clone -q https://github.com/SchafNunes/node21det.git /content/node21det
%cd /content/node21det
!git log --oneline -1
!pip install -q -e .

## 4. Dados no disco local da sessão

Na primeira vez copia `images/` do Drive (lento, milhares de arquivos) e grava
`node21_images.tar` no Drive. Nas próximas sessões copia só o `.tar`.

In [ ]:
import hashlib, os, subprocess, time
LOCAL = "/content/data"
TAR = f"{DRIVE_DATA}/node21_images.tar"
t0 = time.time()
if not os.path.isdir(f"{LOCAL}/images"):
    os.makedirs(LOCAL, exist_ok=True)
    if os.path.exists(TAR):
        subprocess.run(["tar", "-xf", TAR, "-C", LOCAL], check=True)
    else:
        subprocess.run(["cp", "-r", f"{DRIVE_DATA}/images", LOCAL], check=True)
        subprocess.run(["tar", "-cf", TAR, "-C", LOCAL, "images"], check=True)
subprocess.run(["cp", f"{DRIVE_DATA}/metadata.csv", LOCAL], check=True)

n = len([f for f in os.listdir(f"{LOCAL}/images") if f.endswith(".mha")])
md5 = hashlib.md5(open(f"{LOCAL}/metadata.csv", "rb").read()).hexdigest()
print(f"{n} imagens | metadata md5 {md5} | {time.time() - t0:.0f} s")
assert n == 4882, "faltam imagens"
assert md5 == "86c9d97611be7deeca6370cf4ef42503", "metadata.csv diferente do usado nas partições"

## 5. Treino

In [ ]:
OUT = f"{DRIVE_RUNS}/{RUN_NAME}"
!python -W ignore scripts/train.py --config {CONFIG} {EXTRA_ARGS} \
    --images /content/data/images --metadata /content/data/metadata.csv \
    --splits splits/splits.csv --out "{OUT}"

## 6. Resultado

In [ ]:
import pandas as pd
h = pd.read_csv(f"{OUT}/history.csv")
cols = [c for c in ["epoch", "lr", "train_loss", "train_seconds", "gpu_peak_gb", "val_eval_seconds",
                    "val_rank", "val_auc", "val_sens@0.125", "val_sens@0.25", "val_sens@0.5", "val_ap@0.2"] if c in h]
h["train_seconds"] = h["train_seconds"] / 60   # minutos
display(h[cols].round(4))

In [ ]:
if DESCONECTAR_AO_FIM:
    from google.colab import runtime
    runtime.unassign()